In [ ]:
# Assuming this will be run on Google Colab with CPU
from google.colab import drive
drive.mount('/content/drive')

# Please set the path according to the local environment
!ls '/content/drive/MyDrive/pitch_type_pred/'
%cd "/content/drive/MyDrive/pitch_type_pred/"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
dataset  program  results
/content/drive/MyDrive/pitch_type_pred


In [ ]:
import os
import numpy as np
import scipy.io
import glob
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

In [ ]:
# Load dataset from mat files
player_name="sub01" #sub01-08
temp_name="./dataset/"+player_name+"/*.mat"
mat_files = sorted(glob.glob(temp_name))

data_list = []
for f in mat_files:
    mat_data = scipy.io.loadmat(f)
    data_list.append(mat_data)

print(f"The total pitch number {len(data_list)}")

T = data_list[0]["X"].shape[0]
D = data_list[0]["X"].shape[1]
print("Time length:", T, " Number of input features:", D)

The total pitch number 111
Time length: 101  Number of input features: 13


In [ ]:
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class LSTMClassifier(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        _, (h_n, _) = self.lstm(x)  # h_n: (num_layers, B, H)
        h_last = h_n[-1]
        logits = self.fc(h_last)
        return logits.squeeze(1)


def standardize_sequence_train_test(X_train_seq, X_test_seq, eps=1e-8):
    """
    X_train_seq: (N_train, L, D)
    X_test_seq : (N_test,  L, D)

    Standardize per feature using ONLY training data, across (N_train * L).
    """
    Ntr, L, D = X_train_seq.shape
    Xtr_flat = X_train_seq.reshape(-1, D)
    mu = Xtr_flat.mean(axis=0, keepdims=True)
    sd = Xtr_flat.std(axis=0, keepdims=True)
    sd = np.maximum(sd, eps)

    X_train_z = (X_train_seq - mu) / sd
    X_test_z  = (X_test_seq  - mu) / sd
    return X_train_z, X_test_z


# Full-joint condition (same as your setting)
n_train_per_class = 25
n_test_per_class = 5
n_repeats = 100
np.random.seed(0)

# Training hyperparams (you can tune later)
hidden_size = 64
num_layers = 1
lr = 1e-3
epochs = 80
weight_decay = 0.0

acc_means, acc_stds = [], []
for t in range(T):
    print(t)

    # Use sequence from 0..t (length = t+1)
    # X_seq: (N, L, D)
    X_seq = np.array([d["X"][:t+1, :] for d in data_list])
    y = np.array([int(np.squeeze(d["Ball_type_binary"])) for d in data_list])

    accs = []

    for rep in range(n_repeats):
        idx_0 = np.where(y == 0)[0]
        idx_1 = np.where(y == 1)[0]

        train_idx_0 = np.random.choice(idx_0, size=n_train_per_class, replace=False)
        train_idx_1 = np.random.choice(idx_1, size=n_train_per_class, replace=False)

        remain_0 = np.setdiff1d(idx_0, train_idx_0)
        remain_1 = np.setdiff1d(idx_1, train_idx_1)
        test_idx_0 = np.random.choice(remain_0, size=n_test_per_class, replace=False)
        test_idx_1 = np.random.choice(remain_1, size=n_test_per_class, replace=False)

        train_idx = np.concatenate([train_idx_0, train_idx_1])
        test_idx  = np.concatenate([test_idx_0, test_idx_1])

        X_train_seq, y_train = X_seq[train_idx], y[train_idx]
        X_test_seq,  y_test  = X_seq[test_idx],  y[test_idx]

        # Standardize using ONLY train data (avoid information leak)
        X_train_seq, X_test_seq = standardize_sequence_train_test(X_train_seq, X_test_seq)

        # Torch tensors
        X_train_t = torch.tensor(X_train_seq, dtype=torch.float32, device=device)
        y_train_t = torch.tensor(y_train, dtype=torch.float32, device=device)
        X_test_t  = torch.tensor(X_test_seq,  dtype=torch.float32, device=device)
        y_test_t  = torch.tensor(y_test,  dtype=torch.float32, device=device)

        # Model per (t, rep) to match LR setting (no leakage across reps)
        model = LSTMClassifier(input_size=D, hidden_size=hidden_size, num_layers=num_layers).to(device)
        criterion = nn.BCEWithLogitsLoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

        # Train
        model.train()
        for ep in range(epochs):
            optimizer.zero_grad()
            logits = model(X_train_t)
            loss = criterion(logits, y_train_t)
            loss.backward()
            optimizer.step()

        # Test accuracy
        model.eval()
        with torch.no_grad():
            logits = model(X_test_t)
            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).to(torch.int64)
            acc = (preds == y_test_t.to(torch.int64)).float().mean().item()

        accs.append(acc)

    acc_means.append(float(np.mean(accs)))
    acc_stds.append(float(np.std(accs)))
# ====== end replace block ======


0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95


In [ ]:
# Visualize the results
t_range = np.arange(T)

plt.figure(figsize=(10,5))
plt.plot(t_range, acc_means, color='C0', label="Accuracy (mean)")
plt.fill_between(t_range,
                 np.array(acc_means) - np.array(acc_stds),
                 np.array(acc_means) + np.array(acc_stds),
                 color='C0', alpha=0.3)
plt.ylim(0.0, 1.0)
n0, n1 = len(idx_0), len(idx_1)
chance_acc = max(n0, n1) / (n0 + n1)
plt.axhline(0.5, color='gray', linestyle='--', linewidth=1)
plt.xlabel("Normalized time")
plt.ylabel("Accuracy")
temp_name="./results/images/accuracy_plot_lstm_"+player_name+".pdf"
plt.savefig(temp_name, dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
df = pd.DataFrame({
    "t": np.arange(len(acc_means)),
    "acc_mean": acc_means,
    "acc_std": acc_stds,   # acc_stds も保存したい場合
})

csv_path = os.path.join("./results/csv", f"acc_timecourse_lstm_{player_name}.csv")
df.to_csv(csv_path, index=False, encoding="utf-8-sig")

print("Saved:", csv_path)

In [ ]:
from google.colab import runtime
runtime.unassign()